# Lab: prevendo a sobrevivência no Titanic

Notebook de aula da página *Regressão Logística*: https://insper.github.io/ML/pt/2026.2/classes/logistic-regression/

A lista de passageiros do Titanic da aula de [pré-processamento](../preprocessing/index.md), agora com uma pergunta: *quem sobreviveu?* O lab aplica tudo o que foi visto a dados reais e bagunçados: um pipeline com imputação, codificação e escalonamento; probabilidades conferidas à mão; coeficientes lidos como razões de chances; e uma estimativa honesta da acurácia em passageiros que o modelo nunca viu. O modelo é a regressão logística pura desta aula (`C=np.inf`), então não há nada para ajustar: dividimos os dados uma vez em **treino** (ajustar e estudar o modelo) e **teste** (reportar, uma vez, no fim).

Rode as células em ordem. Em cada passo, responda o **Preveja** com a dupla *antes* de rodar. Nas **Simulações**, escreva o experimento na célula vazia; as soluções e a leitura de cada resultado estão na página da aula.

## Passo 1 — carregar e olhar

**Preveja.** A taxa de sobrevivência geral fica acima ou abaixo de 50%? Quem sobreviveu mais: as mulheres da 3ª classe ou os homens da 1ª?

In [ ]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/hsandmann/biblio/refs/heads/main/ml/aula04/titanic/train.csv"
df = pd.read_csv(url)
print(df.shape)
print(f"taxa de sobrevivência: {df['Survived'].mean():.3f}")
print(df.pivot_table(index='Sex', columns='Pclass', values='Survived', aggfunc='mean').round(3))
print("faltantes:", df[['Age', 'Embarked', 'Cabin']].isna().sum().to_dict())

## Passo 2 — treino e teste

**Preveja.** Por que separar o conjunto de teste agora, antes de montar qualquer modelo? Por que não deixar isso para o fim?

In [ ]:
from sklearn.model_selection import train_test_split

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X, y = df[features], df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(len(X_train), len(X_test))
print(f"taxa de sobrevivência: treino {y_train.mean():.3f} · teste {y_test.mean():.3f}")

## Passo 3 — uma baseline

**Preveja.** Um "modelo" que responde sempre "não sobreviveu", sem olhar o passageiro, acerta mais ou menos da metade das vezes? Por quê?

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print(f"sempre 'não sobreviveu': acurácia de treino {dummy.score(X_train, y_train):.3f}")

## Passo 4 — o pipeline

**Preveja.** `Pclass` vale 1, 2 ou 3. Por que tratá-la como categoria (uma coluna one-hot para cada classe) e não como um número?

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric = ['Age', 'SibSp', 'Parch', 'Fare']
categorical = ['Pclass', 'Sex', 'Embarked']

preprocess = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                          OneHotEncoder(drop='first')), categorical)])
model = make_pipeline(preprocess, LogisticRegression(C=np.inf, max_iter=1000))   # C=np.inf: sem penalidade
model.fit(X_train, y_train)

enc = model[0].named_transformers_['cat'][-1]
print("categorias de referência (descartadas):", {c: cats[0] for c, cats in zip(categorical, enc.categories_)})
print(f"{model[-1].coef_.size} coeficientes · acurácia de treino {model.score(X_train, y_train):.3f}")

## Passo 5 — probabilidades para dois passageiros

**Preveja.** Olhando a tabela do passo 1: um homem da 3ª classe e uma mulher da 1ª classe. Quem recebe probabilidade alta e quem recebe baixa?

In [ ]:
passengers = pd.DataFrame({'Pclass': [3, 1], 'Sex': ['male', 'female'], 'Age': [25, 30],
                           'SibSp': [0, 1], 'Parch': [0, 0], 'Fare': [8.05, 80.0],
                           'Embarked': ['S', 'C']})
print(model.predict_proba(passengers).round(3))
print(model.predict(passengers))

**Simulação: o efeito da idade.** Tome um passageiro da 2ª classe que viaja com os pais (`Parch = 2`) e um irmão (`SibSp = 1`), passagem de 30 libras, embarcado em Southampton. Mude só a idade (2, 8, 15, 25, 40 e 60 anos), primeiro para um menino e depois para uma menina. Observe a probabilidade prevista. Depois confira nas linhas de treino quantas crianças de até 12 anos da 2ª classe sobreviveram.

In [ ]:
# sua simulação aqui

## Passo 6 — a probabilidade do homem, à mão

**Preveja.** Olhando os coeficientes que vão aparecer, qual atributo você espera que mais derrube o escore do homem?

In [ ]:
names = model[0].get_feature_names_out()
w, b = model[-1].coef_[0], model[-1].intercept_[0]
x_man = model[0].transform(passengers)[0]            # o homem da 3ª classe, depois do pré-processamento
print(pd.DataFrame({'x': x_man, 'w': w, 'w*x': w * x_man}, index=names).round(3))
z = b + (w * x_man).sum()
print(f"z = {b:.3f} + {(w * x_man).sum():.3f} = {z:.3f}   sigmoid(z) = {1 / (1 + np.exp(-z)):.3f}")

**Simulação: um atributo por vez.** Parta do homem da 3ª classe e transforme-o, um passo de cada vez, na passageira de referência: primeiro mude o sexo para `female`, depois a classe para 1, depois o porto para `C`. A cada passo, imprima o escore \(z\) (`model.decision_function`) e a probabilidade. Quanto \(z\) muda em cada passo? E a probabilidade?

In [ ]:
# sua simulação aqui

## Passo 7 — coeficientes como razões de chances

**Preveja.** O coeficiente de `Age` é negativo. Então, com o resto fixo, um passageiro mais velho tem chance maior ou menor de sobreviver? A razão de chances $e^{\text{coef}}$ dele fica acima ou abaixo de 1?

In [ ]:
coef = pd.Series(w, index=names)
print(pd.DataFrame({'coef': coef, 'e^coef': np.exp(coef)}).round(3))
sd = model[0].named_transformers_['num'][-1].scale_
print("um desvio-padrão:", ", ".join(f"{c} {s:.1f}" for c, s in zip(numeric, sd)))

In [ ]:
print(f"mulher vs homem:  chance x {np.exp(-coef['cat__Sex_male']):.2f}")
print(f"1ª vs 3ª classe:  chance x {np.exp(-coef['cat__Pclass_3']):.2f}")
print(f"2ª vs 3ª classe:  chance x {np.exp(coef['cat__Pclass_2'] - coef['cat__Pclass_3']):.2f}")

**Simulação: a razão de chances é a mesma para todo mundo.** Tome três passageiros diferentes (o homem da 3ª classe, um homem de 30 anos da 1ª classe e o menino da simulação do passo 5). Para cada um, calcule com `predict_proba` a probabilidade como mulher e como homem, mantendo todo o resto. Compare a razão entre as probabilidades e a razão entre as chances.

In [ ]:
# sua simulação aqui

## Passo 8 — o único teste

**Preveja.** A acurácia de teste vai ficar acima ou abaixo dos 80,9% de treino? Muito ou pouco?

In [ ]:
from sklearn.metrics import confusion_matrix

print(f"acurácia de teste {model.score(X_test, y_test):.3f} · baseline {dummy.score(X_test, y_test):.3f}")
print(confusion_matrix(y_test, model.predict(X_test), labels=[1, 0]))   # positivo primeiro

**Simulação: mexendo no limiar.** O `predict` usa o limiar 0,5. Nas linhas de *treino* (o teste já foi usado), classifique com os limiares 0,3, 0,4, 0,5, 0,6 e 0,7 e imprima, para cada um, a fração dos sobreviventes encontrados, a fração dos mortos reconhecidos e a acurácia.

In [ ]:
# sua simulação aqui

**Para fechar a aula.** Em uma frase cada: (1) o que um coeficiente da regressão logística multiplica? (2) Por que o menino da simulação do passo 5 recebeu uma probabilidade tão baixa? (3) Por que o conjunto de teste só pode ser usado uma vez?

## Sua vez

Para casa, com mais tempo:

1. **A interação.** O passo 1 mostrou que a terceira classe prejudica as mulheres muito mais que os homens. Acrescente a coluna `df['female_3rd'] = ((df['Sex'] == 'female') & (df['Pclass'] == 3)).astype(int)`, coloque-a em `features` e em `categorical` (com `drop='first'`, seu coeficiente compara 1 com 0) e refaça os passos 2 a 7. Leia a razão de chances dela em palavras. Como muda o coeficiente `Sex_male`? Por quê? Para decidir se a coluna ajuda, divida `X_train` de novo em uma parte de treino e uma de validação e compare os dois modelos ali, não no teste.
2. **O limiar.** Nessa validação, calcule `p = model.predict_proba(X_val)[:, 1]` e conte os sobreviventes perdidos e os alarmes falsos para os limiares 0,3, 0,5 e 0,7. Que limiar você usaria se perder um sobrevivente fosse três vezes pior que um alarme falso?
3. **Por que uma categoria de referência.** Monte o pipeline de novo com `OneHotEncoder()` (sem `drop`) e ajuste-o. Depois some 5 aos três pesos de `Pclass` em `model[-1].coef_` e subtraia 5 de `model[-1].intercept_`: as probabilidades previstas em `X_test` mudam? O que isso diz sobre ler um único coeficiente quando toda categoria mantém sua coluna?
4. **Do zero.** Tome `Xt = model[0].transform(X_train)` e rode o laço de gradiente descendente desta aula (pesos e intercepto, \(\eta = 0{,}5\)). Compare com `model[-1].coef_` e `model[-1].intercept_`. Depois de quantas iterações eles concordam em duas casas decimais?